# 🌍Small Landscape Features Geospatial Zonal & Temporal Statistics

**Comparing Woody Vegetation Layers (2018 vs 2021) over user-defined Areas of Interest**

---

This notebook guides you through three self-contained geospatial workflows:

| # | Use Case | AOI | Goal |
|---|----------|-----|------|
| A | 🌾 Agriculture | Agricultural parcels | Monitor crop/land indicators per parcel across years |
| B | 🦋 Biodiversity | Administrative units | Aggregate habitat indicators at admin level |
| C | 🏙️ Urban | Urban extents | Track tree canopy extent  |

**Each use case follows the same workflow:**
1. Load & visualise data
2. Select Areas of Interest interactively
3. Run zonal statistics (2018 & 2021)
4. Compute change metrics
5. Visualise results (maps + charts)
6. Export outputs (CSV + GeoJSON)

---
> 📂 **Before running**: Update the file paths in the **Configuration** cell.

## 📋 Table of Contents
- [📦 0 · Imports & Helper Functions](#imports)
- [🌾 Use Case A – Agriculture](#usecaseA)
- [🦋 Use Case B – Biodiversity](#usecaseB)
- [🏙️ Use Case C – Urban](#usecaseC)

<a id='imports'></a>
## 📦 0 · Imports & Helper Functions

In [ ]:
# ── Standard library ──────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show as rshow
from rasterio.mask import mask as rmask
from rasterstats import zonal_stats
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling
)
import xarray as xr
import rioxarray
from rasterio.windows import Window

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from folium.plugins import MeasureControl, MiniMap
import branca.colormap as bcm

# ── Notebook utilities ────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Global style ──────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

print('✅ All libraries imported.')

In [ ]:
# =============================================================
#  GLOBAL CONFIGURATION  –  edit this cell before running
# =============================================================

from pathlib import Path

# ── Root directories ──────────────────────────────────────────
DATA_DIR = Path('data')
OUT_DIR  = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Raster inputs ─────────────────────────────────────────────
RASTER_2018 = DATA_DIR / 'rasters' / 'wvl18_lu_subset.tif'
RASTER_2021 = DATA_DIR / 'rasters' / 'wvl21_lu_subset.tif'

# ── Vector inputs ─────────────────────────────────────────────
AGRI_VECTOR  = DATA_DIR / 'vectors' / 'lpis_lu_2025_subset.gpkg'
ADMIN_VECTOR = DATA_DIR / 'vectors' / 'lpis_lu_2025_subset2.gpkg'
URBAN_VECTOR = DATA_DIR / 'vectors' / 'urban_cities.geojson'

# ── Raster band to analyse (1-indexed) ────────────────────────
BAND_INDEX = 1

# ── Column used as unique identifier in each vector layer ─────
AGRI_ID_COL  = 'FLIK'
ADMIN_ID_COL = 'NUTS_ID'
URBAN_ID_COL = 'city'

# ── Statistics to compute ─────────────────────────────────────
ZONAL_STATS = ['min', 'max', 'mean', 'median', 'std', 'count', 'nodata']

# =============================================================
#  FEATURE 1 · USE-CASE SWITCHER
# =============================================================
# Set to True to activate a use case, False to skip it.
# When False the use-case cells will print a skip message and
# do nothing, so you can safely run the whole notebook.
RUN_USECASE_A = True    # 🌾 Agriculture
RUN_USECASE_B = True    # 🦋 Biodiversity
RUN_USECASE_C = True    # 🏙️ Urban


# =============================================================
#  FEATURE 2 · MAP SYMBOLOGY
# =============================================================
# ── Folium tile background ────────────────────────────────────
# Options: 'CartoDB positron', 'CartoDB dark_matter',
#          'OpenStreetMap', 'Stamen Terrain', 'Stamen Toner'
# MAP_TILES = 'CartoDB positron'
MAP_TILES = folium.TileLayer(
    tiles='https://tiles.stadiamaps.com/tiles/alidade_satellite/{z}/{x}/{y}{r}.{ext}',
    attr='© CNES, Distribution Airbus DS, © Airbus DS, © PlanetObserver '
         '(Contains Copernicus Data) | '
         '© <a href="https://www.stadiamaps.com/" target="_blank">Stadia Maps</a> '
         '© <a href="https://openmaptiles.org/" target="_blank">OpenMapTiles</a> '
         '© <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>',
    min_zoom=0,
    max_zoom=20,
    ext='jpg'
)

# ── Choropleth colormap (matplotlib name) ─────────────────────
# Diverging (good for change maps): 'RdYlGn', 'RdBu', 'PiYG', 'BrBG'
COLORMAP = 'RdYlGn'
CHORO_LOW_COLOR = '#d73027'
CHORO_HIGH_COLOR = '#1a9850'

# ── Raster preview symbology ──────────────────────────────────
# Set RASTER_IS_BINARY = True for 0/1 rasters (custom 2-colour legend).
# Set RASTER_IS_BINARY = False to use a continuous COLORMAP.
RASTER_IS_BINARY = True
RASTER_COLOR_0   = '#8c8c8c'    # colour for value 0
RASTER_COLOR_1   = '#27ae60'    # colour for value 1
RASTER_LABEL_0   = 'Absent (0)'
RASTER_LABEL_1   = 'Present (1)'

# ── Bar chart colours ─────────────────────────────────────────
BAR_COLOR_2018 = '#3498db'   # colour for 2018 bars
BAR_COLOR_2021 = '#e67e22'   # colour for 2021 bars

# ── Change indicator colours (tables & top-change charts) ─────
COLOR_GAIN = '#2ecc71'   # positive change
COLOR_LOSS = '#e74c3c'   # negative change

print('✅ Global configuration loaded.')
print(f'   Raster 2018 : {RASTER_2018}')
print(f'   Raster 2021 : {RASTER_2021}')
print(f'   Output dir  : {OUT_DIR.resolve()}')

In [ ]:
# ================================================================
#  HELPER FUNCTIONS  (shared across all use cases)
# ================================================================

# ── 1. Load & reproject vector to raster CRS ─────────────────
def load_vector(path: Path, raster_path: Path) -> gpd.GeoDataFrame:
    """Load a GeoJSON/shapefile and reproject to the raster CRS."""
    gdf = gpd.read_file(path)
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
    if gdf.crs != raster_crs:
        print(f'  ↪ Reprojecting vector from {gdf.crs} to {raster_crs}')
        gdf = gdf.to_crs(raster_crs)
    return gdf


# ── 2. Run zonal statistics for one raster ───────────────────
def run_zonal_stats(
    gdf: gpd.GeoDataFrame,
    raster_path: Path,
    stats: list,
    band: int = 1,
    prefix: str = ''
) -> gpd.GeoDataFrame:
    """
    Compute zonal statistics for each polygon in gdf against a raster.
    Returns gdf enriched with stat columns prefixed by `prefix`.
    """
    results = zonal_stats(
        vectors=gdf,
        raster=str(raster_path),
        stats=stats,
        band=band,
        nodata=np.nan,
        geojson_out=False
    )
    stats_df = pd.DataFrame(results)
    if prefix:
        stats_df.columns = [f'{prefix}_{c}' for c in stats_df.columns]
    return gdf.reset_index(drop=True).join(stats_df)


# ── 3. Compute change between two years ──────────────────────
def compute_change(
    gdf: gpd.GeoDataFrame,
    stat: str = 'mean',
    year_a: str = '2018',
    year_b: str = '2021'
) -> gpd.GeoDataFrame:
    """
    Add absolute and relative change columns for a given statistic.
    Expects columns named <year_a>_<stat> and <year_b>_<stat>.
    """
    col_a = f'{year_a}_{stat}'
    col_b = f'{year_b}_{stat}'
    gdf[f'delta_{stat}']    = gdf[col_b] - gdf[col_a]
    gdf[f'pct_chg_{stat}']  = ((gdf[col_b] - gdf[col_a]) / gdf[col_a].abs()) * 100
    return gdf


# ── 4. Folium interactive overview map ───────────────────────
def make_overview_map(
    gdf: gpd.GeoDataFrame,
    id_col: str,
    title: str = 'AOI Overview'
) -> folium.Map:
    """Create a Folium map with polygons and popups."""
    gdf_wgs = gdf.to_crs(epsg=4326)
    centroid = gdf_wgs.geometry.unary_union.centroid
    m = folium.Map(
        location=[centroid.y, centroid.x],
        zoom_start=10,
        tiles=MAP_TILES
    )
    folium.GeoJson(
        gdf_wgs.__geo_interface__,
        name='AOI',
        style_function=lambda f: {
            'fillColor': OVERVIEW_FILL_COLOR,
            'color': POLY_EDGE_COLOR,
            'weight': POLY_EDGE_WIDTH,
            'fillOpacity': OVERVIEW_FILL_OPACITY
        },
        tooltip=folium.GeoJsonTooltip(fields=[id_col], aliases=['ID:'])
    ).add_to(m)
    MiniMap(toggle_display=True).add_to(m)
    MeasureControl().add_to(m)
    folium.LayerControl().add_to(m)
    return m

# ----------4b. Folium interactive map with raster 

def display_raster(
    raster_path,
    gdf: gpd.GeoDataFrame,
    id_col: str,
    opacity=0.7,
    zoom_start=12,
    tile_layer=MAP_TILES,
    binary=True,
    subset=False,
    subset_fraction=0.5,
    green_color=(29, 194, 73)
):
    """
    Display a raster in Folium with safe reprojection and binary styling.

    Parameters
    ----------
    raster_path : str or Path
    opacity : float
    zoom_start : int
    tile_layer : str
    binary : bool
        If True → 0 is transparent, 1 is green
    subset : bool
        If True → loads only central subset (faster)
    subset_fraction : float
        Fraction of raster to load (0.5 = 50%)
    green_color : tuple
        RGB for value=1
    """

    ## subset + reproject
    
    with rasterio.open(raster_path) as src:

        if subset:
            w = int(src.width * subset_fraction)
            h = int(src.height * subset_fraction)

            window = Window(
                col_off=(src.width - w) // 2,
                row_off=(src.height - h) // 2,
                width=w,
                height=h
            )

            data = src.read(1, window=window)
            src_transform = src.window_transform(window)

            src_crs = src.crs

        else:
            data = src.read(1)
            src_transform = src.transform
            src_crs = src.crs

        transform, width, height = calculate_default_transform(
            src_crs,
            "EPSG:4326",
            data.shape[1],
            data.shape[0],
            *rasterio.transform.array_bounds(
                data.shape[0], data.shape[1], src_transform
            )
        )

        dst = np.empty((height, width), dtype=np.uint8)

        reproject(
            source=(data > 0).astype(np.uint8),
            destination=dst,
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=transform,
            dst_crs="EPSG:4326",
            resampling=Resampling.nearest
        )

    rgba = np.zeros((height, width, 4), dtype=np.uint8)

    rgba[dst == 1] = [*green_color, 255]  # green opaque
    rgba[dst == 0, 3] = 0                # transparent background
    bounds = rasterio.transform.array_bounds(height, width, transform)

    # ─────────────────────────────────────────────
    # Map
    # ─────────────────────────────────────────────
    gdf_wgs = gdf.to_crs(epsg=4326)
    m = folium.Map(
        location=[(bounds[1] + bounds[3]) / 2,
                  (bounds[0] + bounds[2]) / 2],
        zoom_start=zoom_start,
        tiles=tile_layer
    )

    folium.raster_layers.ImageOverlay(
        image=rgba,
        bounds=[[bounds[1], bounds[0]], [bounds[3], bounds[2]]],
        opacity=opacity,
        name="Raster"
    ).add_to(m)

    folium.GeoJson(
        gdf_wgs.__geo_interface__,
        name='AOI',
        style_function=lambda f: {
            'fillColor': OVERVIEW_FILL_COLOR,
            'color': POLY_EDGE_COLOR,
            'weight': POLY_EDGE_WIDTH,
            'fillOpacity': OVERVIEW_FILL_OPACITY
        },
        tooltip=folium.GeoJsonTooltip(fields=[id_col], aliases=['ID:'])
    ).add_to(m)

    folium.LayerControl().add_to(m)

    return m

def display_raster_bio(
    raster_path,
    gdf: gpd.GeoDataFrame,
    id_col: str,
    opacity=0.7,
    zoom_start=12,
    tile_layer=MAP_TILES,
    binary=True,
    subset=False,
    subset_fraction=0.5,
    green_color=(29, 194, 73)
):
    """
    Display a raster in Folium with safe reprojection and binary styling.

    Parameters
    ----------
    raster_path : str or Path
    opacity : float
    zoom_start : int
    tile_layer : str
    binary : bool
        If True → 0 is transparent, 1 is green
    subset : bool
        If True → loads only central subset (faster)
    subset_fraction : float
        Fraction of raster to load (0.5 = 50%)
    green_color : tuple
        RGB for value=1
    """

    with rasterio.open(raster_path) as src:
        if subset:
            w = int(src.width * subset_fraction)
            h = int(src.height * subset_fraction) // 2

            window = Window(
                col_off=(src.width - w) // 2,
                row_off=(src.height - h) // 2,
                width=w,
                height=h
            )

            data = src.read(1, window=window)
            src_transform = src.window_transform(window)

            src_crs = src.crs

        else:
            data = src.read(1)
            src_transform = src.transform
            src_crs = src.crs

        transform, width, height = calculate_default_transform(
            src_crs,
            "EPSG:4326",
            data.shape[1],
            data.shape[0],
            *rasterio.transform.array_bounds(
                data.shape[0], data.shape[1], src_transform
            )
        )

        dst = np.empty((height, width), dtype=np.uint8)

        reproject(
            source=(data > 0).astype(np.uint8),
            destination=dst,
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=transform,
            dst_crs="EPSG:4326",
            resampling=Resampling.nearest
        )

    rgba = np.zeros((height, width, 4), dtype=np.uint8)

    rgba[dst == 1] = [*green_color, 255]  # green opaque
    rgba[dst == 0, 3] = 0                # transparent background

    bounds = rasterio.transform.array_bounds(height, width, transform)

    # ─────────────────────────────────────────────
    # Map
    # ─────────────────────────────────────────────
    gdf_wgs = gdf.to_crs(epsg=4326)
    m = folium.Map(
        location=[(bounds[1] + bounds[3]) / 2,
                  (bounds[0] + bounds[2]) / 2],
        zoom_start=zoom_start,
        tiles=tile_layer
    )

    folium.raster_layers.ImageOverlay(
        image=rgba,
        bounds=[[bounds[1], bounds[0]], [bounds[3], bounds[2]]],
        opacity=opacity,
        name="Raster"
    ).add_to(m)

    folium.GeoJson(
        gdf_wgs.__geo_interface__,
        name='AOI',
        style_function=lambda f: {
            'fillColor': OVERVIEW_FILL_COLOR,
            'color': POLY_EDGE_COLOR,
            'weight': POLY_EDGE_WIDTH,
            'fillOpacity': OVERVIEW_FILL_OPACITY
        },
        tooltip=folium.GeoJsonTooltip(fields=[id_col], aliases=['ID:'])
    ).add_to(m)

    folium.LayerControl().add_to(m)

    return m


# ── 5a. Apply delta filter ────────────────────────────────────
def apply_delta_filter(
    gdf: gpd.GeoDataFrame,
    stat: str = 'mean'
) -> gpd.GeoDataFrame:
    """
    Filter polygons for map display based on the DELTA_FILTER config variable.
    Does NOT modify the underlying dataset — only affects what is drawn on maps.
    """
    col = f'delta_{stat}'
    n_before = len(gdf)
    if DELTA_FILTER == 'positive':
        gdf = gdf[gdf[col] > 0]
    elif DELTA_FILTER == 'negative':
        gdf = gdf[gdf[col] < 0]
    elif DELTA_FILTER == 'nonzero':
        gdf = gdf[gdf[col] != 0]
    elif DELTA_FILTER == 'custom':
        gdf = gdf[(gdf[col] >= DELTA_MIN) & (gdf[col] <= DELTA_MAX)]
    # 'all' → no filtering
    print(f'   Delta filter [{DELTA_FILTER}]: {n_before} → {len(gdf)} polygons')
    return gdf


# ── 5b. Apply map polygon limit ───────────────────────────────
def apply_map_limit(gdf: gpd.GeoDataFrame, sort_col: str = None) -> gpd.GeoDataFrame:
    """
    Cap the number of polygons rendered on a map to MAP_MAX_POLYGONS.
    If sort_col is provided, keeps the top-N by absolute value of that column.
    Otherwise keeps the first N rows.
    """
    if MAP_MAX_POLYGONS is None or len(gdf) <= MAP_MAX_POLYGONS:
        return gdf
    if sort_col and sort_col in gdf.columns:
        gdf = gdf.reindex(gdf[sort_col].abs().nlargest(MAP_MAX_POLYGONS).index)
    else:
        gdf = gdf.head(MAP_MAX_POLYGONS)
    print(f'   Map limit: showing {len(gdf)} of {MAP_MAX_POLYGONS} max polygons')
    return gdf


# ── 5c. Choropleth change map ─────────────────────────────────

def make_choropleth_map(
    gdf: gpd.GeoDataFrame,
    value_col: str,
    id_col: str,
    title: str = 'Choropleth Map',
    stat: str = 'mean',
    apply_filters: bool = True,
    low_color = CHORO_LOW_COLOR,
    high_color = CHORO_HIGH_COLOR,
    legend_title: str = None
) -> folium.Map:
    """
    Continuous choropleth map with user-defined min/max colors.

    The color gradient is automatically interpolated between:
      - low_color → minimum value
      - high_color → maximum value
    """

    gdf = gdf.copy()

    # ── Apply global filters ───────────────────────────────────
    if apply_filters:
        gdf = apply_delta_filter(gdf, stat=stat)
        gdf = apply_map_limit(gdf, sort_col=value_col)

    if gdf.empty:
        print('⚠️ No polygons to display.')
        return folium.Map(tiles=MAP_TILES)

    # ── Reproject for Folium ──────────────────────────────────
    gdf_wgs = gdf.to_crs(epsg=4326)
    gdf_wgs[id_col] = gdf_wgs[id_col].astype(str)

    centroid = gdf_wgs.geometry.unary_union.centroid

    m = folium.Map(
        location=[centroid.y, centroid.x],
        zoom_start=10,
        tiles=MAP_TILES
    )

    values = gdf_wgs[value_col].dropna()
    vmin, vmax = values.min(), values.max()

    # ── Continuous custom gradient ────────────────────────────
    colorscale = bcm.LinearColormap(
        colors=[low_color, high_color],
        vmin=vmin,
        vmax=vmax,
        caption=legend_title or title
    )
    colorscale.add_to(m)

    def style_fn(feature):
        val = feature['properties'].get(value_col)
        return {
            'fillColor': colorscale(val) if val is not None else '#cccccc',
            'color': POLY_EDGE_COLOR,
            'weight': POLY_EDGE_WIDTH,
            'fillOpacity': POLY_FILL_OPACITY
        }

    # ── Add polygons ──────────────────────────────────────────
    folium.GeoJson(
        gdf_wgs.__geo_interface__,
        name=title,
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=[id_col, value_col],
            aliases=['ID:', f'{legend_title or title}:'],
            localize=True
        )
    ).add_to(m)

    MiniMap(toggle_display=True).add_to(m)
    folium.LayerControl().add_to(m)

    return m



# ── 6. Summary statistics table ──────────────────────────────
def display_stats_table(
    gdf: gpd.GeoDataFrame,
    id_col: str,
    stat: str = 'mean'
):
    """Display a styled HTML comparison table."""
    cols = [id_col, f'2018_{stat}', f'2021_{stat}', f'delta_{stat}']
    sub = gdf[cols].copy().round(3)
    sub.columns = [id_col, f'{stat.capitalize()} 2018', f'{stat.capitalize()} 2021',
                   'Δ Absolute']

    def colour_delta(v):
        if pd.isna(v): return ''
        c = '#2ecc71' if v > 0 else '#e74c3c' if v < 0 else ''
        return f'color: {c}; font-weight: bold'

    styled = (
        sub.style
        .applymap(colour_delta, subset=['Δ Absolute'])
        .format({'Δ Absolute': '{:+.3f}'})
        .background_gradient(cmap='RdYlGn', subset=[f'{stat.capitalize()} 2018', f'{stat.capitalize()} 2021'])
        .set_table_styles([{'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white')]}])
        .set_caption(f'Zonal Statistics Comparison – {stat.upper()}')
    )
    display(styled)


# ── 7. Grouped bar chart ──────────────────────────────────────
def plot_bar_comparison(
    gdf: gpd.GeoDataFrame,
    id_col: str,
    stat: str = 'mean',
    title: str = 'Comparison',
    ylabel: str = 'Value',
    n: int = 20,
    start: int = 0
):
    """
    Grouped bar chart comparing 2018 vs 2021 for a slice of features.

    Parameters
    ----------
    start : int
        Start index of the slice (0-based).
    n : int
        Number of features to display.
    """
    cols = [id_col, f'2018_{stat}', f'2021_{stat}']
    sub = (
        gdf[cols]
        .dropna()
        .iloc[start:start + n]
    )

    if sub.empty:
        raise ValueError("Selected range is empty. Check 'start' and 'n'.")

    x = np.arange(len(sub))
    w = 0.38

    fig, ax = plt.subplots(figsize=(max(10, len(sub) * 0.55), 5))
    ax.bar(x - w / 2, sub[f'2018_{stat}'], w, label='2018',
           color='#3498db', alpha=0.85)
    ax.bar(x + w / 2, sub[f'2021_{stat}'], w, label='2021',
           color='#e67e22', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(sub[id_col].astype(str),
                       rotation=45, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend()

    plt.tight_layout()
    plt.show()


# ── 8. Change distribution plot ─────────────────────────────
def plot_change_distribution(
    gdf: gpd.GeoDataFrame,
    stat: str = 'mean',
    title: str = 'Distribution of Change'
):
    """KDE + histogram of absolute change values."""
    delta_col = f'delta_{stat}'
    vals = gdf[delta_col].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram
    axes[0].hist(vals, bins=30, color='#9b59b6', edgecolor='white', alpha=0.8)
    axes[0].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_xlabel(f'Δ {stat}')
    axes[0].set_title('Distribution of Absolute Change')

    # Scatter 2018 vs 2021
    axes[1].scatter(gdf[f'2018_{stat}'], gdf[f'2021_{stat}'], alpha=0.6, edgecolors='grey', linewidths=0.3)
    lims = [
        min(gdf[f'2018_{stat}'].min(), gdf[f'2021_{stat}'].min()),
        max(gdf[f'2018_{stat}'].max(), gdf[f'2021_{stat}'].max())
    ]
    axes[1].plot(lims, lims, 'r--', linewidth=1, label='No change')
    axes[1].set_xlabel(f'{stat} 2018')
    axes[1].set_ylabel(f'{stat} 2021')
    axes[1].set_title('2018 vs 2021 Scatter')
    axes[1].legend()

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ── 09. Export function ───────────────────────────────────────
def export_results(
    gdf: gpd.GeoDataFrame,
    name: str,
    out_dir: Path = OUT_DIR
):
    """Export enriched GeoDataFrame as CSV and GeoJSON."""
    # Drop geometry for CSV
    df = pd.DataFrame(gdf.drop(columns='geometry'))
    csv_path  = out_dir / f'{name}_stats.csv'
    geoj_path = out_dir / f'{name}_stats.geojson'

    df.to_csv(csv_path, index=False)
    gdf.to_crs(epsg=4326).to_file(geoj_path, driver='GeoJSON')

    print(f'💾 Exported:')
    print(f'   CSV     → {csv_path}')
    print(f'   GeoJSON → {geoj_path}')
    return csv_path, geoj_path


# ── 11. Dataset-level summary table ──────────────────────────
def display_summary_table(
    gdf: gpd.GeoDataFrame,
    stat: str = 'mean'
):
    """
    Display a compact summary of key statistics across all polygons
    for both years and the change between them.
    Rows = metrics (mean, median, min, max, std, count).
    Columns = 2018 value / 2021 value / absolute change / % change.
    """
    col18, col21 = f'2018_{stat}', f'2021_{stat}'
    v18 = gdf[col18].dropna()
    v21 = gdf[col21].dropna()

    metrics = ['mean', 'median', 'min', 'max', 'std', 'count']
    rows = []
    for m in metrics:
        if m == 'count':
            val18 = v18.count()
            val21 = v21.count()
        else:
            val18 = getattr(v18, m)()
            val21 = getattr(v21, m)()
        delta    = val21 - val18
        rows.append({'Metric': m.capitalize(), '2018': val18, '2021': val21,
                     'Δ Absolute': delta})

    summary = pd.DataFrame(rows).set_index('Metric').round(4)

    def colour_delta(v):
        if pd.isna(v): return ''
        c = '#2ecc71' if v > 0 else '#e74c3c' if v < 0 else ''
        return f'color: {c}; font-weight: bold'

    styled = (
        summary.style
        .applymap(colour_delta, subset=['Δ Absolute'])
        .format( {'Δ Absolute': '{:+.4f}'}, na_rep='—')
        .background_gradient(cmap='Blues', subset=['2018', '2021'])
        .set_table_styles([{'selector': 'th', 'props': [
            ('background-color', '#2c3e50'), ('color', 'white'), ('font-size', '12px')
        ]}])
        .set_caption(f'Dataset Summary – zonal {stat.upper()} across all polygons')
    )
    display(styled)
    return summary


print('✅ Helper functions defined.')

---
<a id='usecaseA'></a>
# 🌾 Use Case A – Agriculture

**Objective:** Quantify raster-derived indicators (e.g. vegetation index, land cover value) over individual agricultural parcels and track their evolution between 2018 and 2021.

**Typical questions:**
- Which parcels showed the largest increase or decrease in the indicator?
- How did the distribution of values shift across the study area?

**What will you see ?**

Woody Vegetation Layer is a binary layer that display presence (pixel value =1) / absence (pixel value =0) of woody vegetation.  
Among the available statistics that will be presented, the "Mean" statistic thus reflects, per polygon, the percentage of woody vegetation presence within the polygon. Ex: mean = 4.88 for all AOI indicates that across the whole AOI, polygons contains on average 4.88% of woody vegetation.
"Delta mean" between 2018 and 2021 then reflects the shift (if any) between the 2 reference years

### A.0 Area of Interest Configuration

3 options: Ireland, Luxemburg, France

In [ ]:

#AOI choice: France, Ireland, Luxemburg

AOI = 'IE' # update here with ISO code of country : FR, IE, LU

if AOI == 'LU':
    RASTER_2018 = DATA_DIR / 'rasters' / 'wvl18_ag_lu_subset.tif'
    RASTER_2021 = DATA_DIR / 'rasters' / 'wvl21_ag_lu_subset.tif'
    AGRI_VECTOR  = DATA_DIR / 'vectors' / 'agr_parcels_LU.gpkg'
    AGRI_ID_COL  = 'FLIK'
elif AOI == 'FR':
    RASTER_2018 = DATA_DIR / 'rasters' / 'wvl18_ag_fr_subset.tif'
    RASTER_2021 = DATA_DIR / 'rasters' / 'wvl21_ag_fr_subset.tif'
    AGRI_VECTOR  = DATA_DIR / 'vectors' / 'agr_parcels_FR.gpkg'
    AGRI_ID_COL  = 'id_parcel'
elif AOI == 'IE':
    RASTER_2018 = DATA_DIR / 'rasters' / 'wvl18_ag_ie_subset.tif'
    RASTER_2021 = DATA_DIR / 'rasters' / 'wvl21_ag_ie_subset.tif'
    AGRI_VECTOR  = DATA_DIR / 'vectors' / 'agr_parcels_IE.gpkg'
    AGRI_ID_COL  = 'FLIK'
else:
    print('Error in AOI definition')
    

### A.1 · Load Data

In [ ]:
print('📂 Loading agricultural parcels...')
agri_gdf = load_vector(AGRI_VECTOR, RASTER_2018)

print(f'   {len(agri_gdf)} parcels loaded')
print(f'   CRS   : {agri_gdf.crs}')
print(f'   Columns: {list(agri_gdf.columns)}')
agri_gdf.head()

### A.2 · Visualise Study Area

In [ ]:
# Interactive Folium overview map
# ── Polygon border colour & width ─────────────────────────────
POLY_EDGE_COLOR = '#fcba03'  # <-- border colour (hex or CSS name)
POLY_EDGE_WIDTH = 0.8        # <-- border width in pixels
POLY_FILL_OPACITY = 0.75     # <-- fill transparency (0=transparent, 1=opaque)

# ── Overview map fill colour (before stats are computed) ──────
OVERVIEW_FILL_COLOR   = '#3186cc'
OVERVIEW_FILL_OPACITY = 0.1   # lower the number, higher the transparency
subset = 0.4                  # For faster map generation, load only a part of the WVL product

m = display_raster(
    RASTER_2021,
    agri_gdf,
    AGRI_ID_COL,
    subset=True,
    subset_fraction=subset,
    zoom_start=13
)
m

### A.3 · Zonal Statistics

Due to large number of polygons, following zonal statistics have been pre-computed for each reference years and for each polygons, taking into account all WVL pixels intersecting said polygons
1. min
2. max 
3. mean 
4. median
5. std
6. count


In [ ]:
agri_sub = agri_gdf.copy()

print(f'✅ Zonal statistics loaded — {len(agri_sub)} parcels processed')


### A.4 · Results Summary

In [ ]:
# Dataset-level summary: key stats across all parcels for each year
# Please choose among the previous listed statistics 'min', 'max', 'mean', 'median','std', 'count'
agri_summary = display_summary_table(agri_sub, stat='mean')

### A.5 · Visualisations

In [ ]:
# As above, you can choose among the following statistics to be displayed for a subset of the polygons
#'min', 'max', 'mean', 'median','std', 'count'

stat = 'mean'
# each area of interest has 10 000+ parcels, difficult to display all of them in one histogram plot. The following options help define which subset you will display
start = 1 # first polygon to be displayed in plot. 
range= 20 # number of polygons to display in chart. 

plot_bar_comparison(
    agri_sub,
    id_col=AGRI_ID_COL,
    stat= stat,
    title=f'Agriculture – {stat} Raster Value per Parcel (2018 vs 2021) - Subset',
    ylabel=f'{stat} Value',
    start= start,
    n= range
)


In [ ]:
#You can choose among the following statistics to be displayed for a subset of the polygons
#'mean', 'median'

stat = 'mean'

plot_change_distribution(
    agri_sub,
    stat=stat,
    title=f'Agriculture – Distribution of Change in {stat} Value'
)

In [ ]:
# Map of absolute change

# Maximum number of polygons rendered on change maps.
# Set to None to display all polygons.
MAP_MAX_POLYGONS = None   # <-- e.g. 200, 500, None

# Controls which polygons appear on change maps.
# Options:
#   'all'      → show every polygon
#   'positive' → only polygons where delta > 0  (gains)
#   'negative' → only polygons where delta < 0  (losses)
#   'nonzero'  → exclude polygons where delta == 0
#   'custom'   → apply DELTA_MIN / DELTA_MAX thresholds below
DELTA_FILTER = 'all'     # <-- change to filter what is shown on maps

# Used only when DELTA_FILTER = 'custom'
DELTA_MIN =  -0.5         # <-- minimum delta to display (inclusive)
DELTA_MAX =  0.0         # <-- maximum delta to display (inclusive)

# ── Polygon border colour & width ─────────────────────────────
POLY_EDGE_COLOR = '#000000'  # <-- border colour (hex or CSS name)
POLY_EDGE_WIDTH = 1       # <-- border width in pixels
POLY_FILL_OPACITY = 0.75     # <-- fill transparency (0=transparent, 1=opaque)
low_color = '#ff002f'       # <-- fill colour for lower value of delta (hex or CSS name)
high_color = '#e9f564'      # <-- fill colour for higher value of delta (hex or CSS name)


agri_change_map = make_choropleth_map(
    agri_sub,
    value_col='delta_mean',
    id_col=AGRI_ID_COL,
    low_color = low_color,
    high_color = high_color,
    title='Δ Mean Value 2018→2021 (Agriculture)'
)
display(agri_change_map)

In [ ]:
# Boxplot: distribution of mean values per year
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(
    [agri_sub['2018_mean'].dropna(), agri_sub['2021_mean'].dropna()],
    labels=['2018', '2021'],
    patch_artist=True,
    boxprops=dict(facecolor='#3498db', alpha=0.7),
    medianprops=dict(color='black', linewidth=2)
)
ax.set_ylabel('Mean Raster Value')
ax.set_title('Agriculture – Distribution of Parcel Mean Values')
plt.tight_layout()
plt.show()

### A.6 · Export Results

In [ ]:
csv_a, geoj_a = export_results(agri_sub, name=f'usecase_agriculture_{AOI}')

---
<a id='usecaseB'></a>
# 🦋 Use Case B – Biodiversity

**Objective:** Aggregate Woody Vegetation Layers indicators at administrative unit level, identifying spatial patterns and temporal trends between 2018 and 2021. This use case focus on Italy's provinces (NUTS level 3)

**Typical questions:**
- Which administrative units show the highest habitat quality?
- Where did conditions improve or deteriorate?

**What will you see ?**

Woody Vegetation Layer is a binary layer that display presence (pixel value =1) / absence (pixel value =0) of woody vegetation.  
Among the available statistics that will be presented, the "Mean" statistic thus reflects, per polygon, the percentage of woody vegetation presence within the polygon. Ex: mean = 4.88 for all AOI indicates that across the whole AOI, polygons contains on average 4.88% of woody vegetation.
"Delta mean" between 2018 and 2021 then reflects the shift (if any) between the 2 reference years


### B.0 . Configuration


In [ ]:
RASTER_2018 = DATA_DIR / 'rasters' / 'wvl18_nutsIT.tif'
RASTER_2021 = DATA_DIR / 'rasters' / 'wvl21_nutsIT.tif'
ADMIN_VECTOR = DATA_DIR / 'vectors' / 'nuts3_IT.gpkg'
ADMIN_ID_COL  = 'NUTS_ID'

### B.1 · Load Data

In [ ]:
print('📂 Loading administrative units...')
admin_gdf = load_vector(ADMIN_VECTOR, RASTER_2018)

print(f'   {len(admin_gdf)} units loaded')
print(f'   CRS     : {admin_gdf.crs}')
print(f'   Columns : {list(admin_gdf.columns)}')
admin_gdf.head()

In [ ]:
# Interactive Folium overview map
# ── Polygon border colour & width ─────────────────────────────
POLY_EDGE_COLOR = '#fcba03'  # <-- border colour (hex or CSS name)
POLY_EDGE_WIDTH = 0.8        # <-- border width in pixels
POLY_FILL_OPACITY = 0.75     # <-- fill transparency (0=transparent, 1=opaque)

# ── Overview map fill colour (before stats are computed) ──────
OVERVIEW_FILL_COLOR   = '#3186cc'
OVERVIEW_FILL_OPACITY = 0.1   # lower the number, higher the transparency

# For faster map generation, load only a part of the WVL product: 0.05 = 5%
# This use case concerns a whole country (Italy), so please refrain from using high subset unless you want to spend a long time waiting for the map to be displayed. Defaut : 0.05
subset = 0.05                 

m = display_raster_bio(
    RASTER_2021,
    admin_gdf,
    ADMIN_ID_COL,
    subset=True,
    subset_fraction=subset,
    zoom_start=12
)
m

### B.2 · Visualise Study Area

### B.3 · Zonal Statistics

In [ ]:
## As the aera of interest is quite large is this use case, statistics have been pre-computed already. We only load them.

admin_sub = admin_gdf.copy()

print(f'✅ Zonal statistics loaded — {len(admin_sub)} parcels processed')

### B.4 · Results Table

In [ ]:
# Dataset-level summary: key stats across all provinces for each year
# Please choose among the previous listed statistics 'min', 'max', 'mean', 'median','std', 'count'

display_stats_table(admin_sub, id_col=ADMIN_ID_COL, stat='mean')

### B.5 · Visualisations

In [ ]:
# As above, you can choose among the following statistics to be displayed for a subset of the polygons
#'min', 'max', 'mean', 'median','std', 'count'

stat = 'mean'

# There are 107 NUTS3 region in Italy, displaying all of them in one histogram plot may be difficult to read. The following options help define which subset you will display
start = 1 # first region to be displayed in plot (they are sort by alphabetical order of their unique NUTS identifier. Ex: Roma : ITI43
range= 10 # number of provinces to display in chart

plot_bar_comparison(
    admin_sub,
    id_col=ADMIN_ID_COL,
    stat=stat,
    title=f'Biodiversity – {stat} Value per Admin Unit (2018 vs 2021)',
    ylabel=f'{stat} Value',
    start= start,
    n= range
)


In [ ]:
# Change map

# Maximum number of polygons rendered on change maps.
# Set to None to display all polygons.
MAP_MAX_POLYGONS = None   # <-- e.g. 200, 500, None

# Controls which polygons appear on choropleth change maps.
# Options:
#   'all'      → show every polygon
#   'positive' → only polygons where delta > 0  (gains)
#   'negative' → only polygons where delta < 0  (losses)
#   'nonzero'  → exclude polygons where delta == 0
#   'custom'   → apply DELTA_MIN / DELTA_MAX thresholds below
DELTA_FILTER = 'all'     # <-- change to filter what is shown on maps

# Used only when DELTA_FILTER = 'custom'
DELTA_MIN =  -0.5         # <-- minimum delta to display (inclusive)
DELTA_MAX =  0.0         # <-- maximum delta to display (inclusive)

# ── Polygon border colour & width ─────────────────────────────
POLY_EDGE_COLOR = '#000000'  # <-- border colour (hex or CSS name)
POLY_EDGE_WIDTH = 1       # <-- border width in pixels
POLY_FILL_OPACITY = 0.75     # <-- fill transparency (0=transparent, 1=opaque)
low_color = '#ff002f'       # <-- fill colour for lower value of delta (hex or CSS name)
high_color = '#e9f564'      # <-- fill colour for higher value of delta (hex or CSS name)

biodiv_change_map = make_choropleth_map(
    admin_sub,
    value_col='delta_mean',
    id_col=ADMIN_ID_COL,
    title='Δ Mean Value 2018→2021 (Biodiversity)'
)
display(biodiv_change_map)

In [ ]:
#You can choose among the following statistics to be displayed for a subset of the polygons
#'mean', 'median'
stat = 'mean'

plot_change_distribution(
    admin_sub,
    stat=stat,
    title=f'Biodiversity – Distribution of Change in {stat} Value'
)

### B.7 · Export Results

In [ ]:
csv_b, geoj_b = export_results(admin_sub, name='usecaseB_biodiversity')

---
<a id='usecaseC'></a>
# 🏙️ Use Case C – Urban

**Objective:** Quantify raster-derived urban indicators (tree canopy extent) over urban extent polygons, comparing the 2018 and 2021 states to reveal patterns.

**Typical questions:**
- What is the tree presence evolution between 2018 and 2021 in my city ?
- How does it compare with other cities?

**Area of Interest**

3 cities are considered here: Linz (AT), Reggio Emilia (IT) and London (UK).
Their respective urban extent is derived from Urban areas of the CLMS Urban Atlas product.

**What will you see ?**

Woody Vegetation Layer is a binary layer that display presence (pixel value =1) / absence (pixel value =0) of woody vegetation.  
Among the available statistics that will be presented, the "Mean" statistic thus reflects, per polygon, the percentage of woody vegetation presence within the polygon. Ex: mean = 4.88 for all AOI indicates that across the whole AOI, polygons contains on average 4.88% of woody vegetation.
"Delta mean" between 2018 and 2021 then reflects the shift (if any) between the 2 reference years


### C.0. Configuration

In [ ]:

# ─────────────────────────────────────────────────────────────
# Urban multi-city configuration
# ─────────────────────────────────────────────────────────────

URBAN_CITIES = {
    'Reggio Emilia': {
        'vector': DATA_DIR / 'vectors' / 'urban_emilia_core.gpkg',
        'raster_2018': DATA_DIR / 'rasters' / 'wvl18_ur_emilia.tif',
        'raster_2021': DATA_DIR / 'rasters' / 'wvl21_ur_emilia.tif',
    },
    'London': {
       'vector': DATA_DIR / 'vectors' / 'urban_london_core.gpkg',
       'raster_2018': DATA_DIR / 'rasters' / 'wvl18_ur_london.tif',
       'raster_2021': DATA_DIR / 'rasters' / 'wvl21_ur_london.tif',
    },
    'Linz': {
        'vector': DATA_DIR / 'vectors' / 'urban_linz_core.gpkg',
        'raster_2018': DATA_DIR / 'rasters' / 'wvl18_ur_linz.tif',
        'raster_2021': DATA_DIR / 'rasters' / 'wvl21_ur_linz.tif',
    }
}

print(f'✅ {len(URBAN_CITIES)} urban cities configured')

### C.1 · Per City Zonal Statistics

Computation of statistics for this use case are much faster than for the previous ones. They can be computed on the fly during this step.

In [ ]:
urban_results = {}  
urban_tables = []   

for city, cfg in URBAN_CITIES.items():
    print(f'\n⏳ Processing {city}')

    # Load and reproject vector
    gdf = load_vector(cfg['vector'], cfg['raster_2018'])
    gdf['city'] = city

    sub = gdf.copy()

    # Zonal stats
    sub = run_zonal_stats(
        sub, cfg['raster_2018'],
        ZONAL_STATS, band=BAND_INDEX, prefix='2018'
    )
    sub = run_zonal_stats(
        sub, cfg['raster_2021'],
        ZONAL_STATS, band=BAND_INDEX, prefix='2021'
    )

    # Temporal change
    sub = compute_change(sub, stat='mean')
    sub = compute_change(sub, stat='median')

    urban_results[city] = sub
    urban_tables.append(sub.drop(columns='geometry'))

    print(f'✅ {city}: {len(sub)} urban polygons processed')


### C.2 · Visualise Study Area

In [ ]:
# ── Map styling (shared across all cities) ──────────────────────────────────────

# ── Polygon border colour & width ─────────────────────────────
POLY_EDGE_COLOR = '#ff0000'   # <-- border colour (hex or CSS name)
POLY_EDGE_WIDTH = 1           # <-- border width in pixels
POLY_FILL_OPACITY = 0.75      # <-- fill transparency (0=transparent, 1=opaque)
# ── Overview map fill colour (before stats are computed) ──────
OVERVIEW_FILL_COLOR = '#3186cc'
OVERVIEW_FILL_OPACITY = 0.3
# For faster map generation, load only a part of the WVL product 0.4 = 40%
subset = 0.4                  



for city, gdf in urban_results.items():
    print(f'Overview map — {city}')

    cfg = URBAN_CITIES[city]

    m = display_raster(
        raster_path=cfg['raster_2021'],
        gdf=gdf,
        id_col=URBAN_ID_COL,
        subset=True,
        subset_fraction=subset,
        zoom_start=13
    )

    display(m)

### C.3 · Results Table

In [ ]:
#Available statistics : 'min', 'max', 'mean', 'median','std', 'count'
stat = 'mean'

for city, sub in urban_results.items():
    print(f'📊 Statistics table — {city}')
    display_stats_table(sub, id_col=URBAN_ID_COL, stat=stat)

### C.4 · Visualisations

In [ ]:
# you can choose among the following statistics to be displayed for a subset of the polygons
#'min', 'max', 'mean', 'median','std', 'count'

stat = 'mean'

for city, sub in urban_results.items():
    plot_bar_comparison(
        sub,
        id_col=URBAN_ID_COL,
        stat=stat,
        title=f'{city} – {stat} Value per Urban Zone (2018 vs 2021)',
        ylabel=f'{stat} Raster Value'
    )

In [ ]:
#You can choose among the following statistics to be displayed for a subset of the polygons
#'mean', 'median'

stat = 'mean'

for city, sub in urban_results.items():
    plot_change_distribution(
        sub,
        stat=stat,
        title=f'Agriculture – Distribution of Change in {stat} Value - {city}'
    )




### C.5 · Change Maps

In [ ]:
# Maps of absolute change

MAP_MAX_POLYGONS = None   # <-- e.g. 200, 500, None

# Controls which polygons appear on choropleth change maps.
# Options:
#   'all'      → show every polygon
#   'positive' → only polygons where delta > 0  (gains)
#   'negative' → only polygons where delta < 0  (losses)
#   'nonzero'  → exclude polygons where delta == 0
#   'custom'   → apply DELTA_MIN / DELTA_MAX thresholds below
DELTA_FILTER = 'all'     # <-- change to filter what is shown on maps

# Used only when DELTA_FILTER = 'custom'
DELTA_MIN =  -0.5         # <-- minimum delta to display (inclusive)
DELTA_MAX =  0.0         # <-- maximum delta to display (inclusive)

# ── Polygon border colour & width ─────────────────────────────
POLY_EDGE_COLOR = '#000000'  # <-- border colour (hex or CSS name)
POLY_EDGE_WIDTH = 1       # <-- border width in pixels
POLY_FILL_OPACITY = 0.75     # <-- fill transparency (0=transparent, 1=opaque)
low_color = '#ff002f'       # <-- fill colour for lower value of delta (hex or CSS name)
high_color = '#e9f564'      # <-- fill colour for higher value of delta (hex or CSS name)


for city, sub in urban_results.items():
    print(f'🗺️ Change map — {city}')
    display(
        make_choropleth_map(
            sub,
            value_col='delta_mean',
            id_col=URBAN_ID_COL,
            title=f'{city} – Δ Mean Value 2018→2021'
        )
    )

### C.6 Cross City Comparison

In [ ]:
#Display main results in one common table
urban_all = pd.concat(urban_tables, ignore_index=True)

summary_city = (
    urban_all
    .groupby('city')[['2018_mean', '2021_mean', 'delta_mean']]
    .mean()
    .round(3)
)

display(summary_city)


In [ ]:
#Display main results in one common plot

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in summary_city['delta_mean']]

ax.bar(summary_city.index, summary_city['delta_mean'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean Δ Value')
ax.set_title('Urban – Mean Change per City (2018→2021)')
plt.tight_layout()
plt.show()

### C.7 · Export Results

#### Per city

In [ ]:
for city, sub in urban_results.items():
    export_results(sub, name=f'usecaseC_urban_{city.lower()}')

#### All cities export (csv only)

In [ ]:
urban_all.to_csv(
    OUT_DIR / 'usecaseC_urban_all_cities_stats.csv',
    index=False
)

print('💾 Combined urban city results exported')

---

## ✅ All Done!

All outputs have been saved to the `outputs/` directory:

